# 1D J1J2 Inference: LorentzGRU  (seed 111-555) - part 2

This is part of the work arxiv:2604.24337 [quant-ph] by HL Dao. For the purpose of reproducing the results, the paths to the trained weights have to be adjusted to match the repo structure.

In this notebook, we performed inference for J2=0.8 case. In another notebook, the inferences for J2=0.0,0.2, 0.5 were done. 

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys
import time
sys.path.append('../../1_hypnqs_torch_lorentz/utility_lorentz')
from j1j2_train_loop_lorentz import *

Hypercore Lorentzian module loaded successfully with Geoopt wrappers.
GPU Available:  False


In [2]:
def set_cpu_deterministic(seed):
    # 1. Python & Numpy
    random.seed(seed)
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    
    # 2. PyTorch CPU
    torch.manual_seed(seed)
    
    # 3. Force Deterministic Algorithms
    # This prevents non-deterministic CPU operations (like some views/reductions)
    torch.use_deterministic_algorithms(True)
    
    # 4. Limit CPU Threads
    # Setting this to 1 ensures operations are done in a fixed order.
    torch.set_num_threads(1)

In [2]:
def clip_local_energies(eloc, threshold=5.0):
    # Convert to numpy if it's a torch tensor, or vice versa
    eloc_real = np.real(eloc)
    median = np.median(eloc_real)
    mad = np.median(np.abs(eloc_real - median))
    
    # Standard safety check to avoid division by zero if MAD is 0
    if mad == 0:
        return eloc
        
    lower_bound = median - threshold * mad
    upper_bound = median + threshold * mad
    
    # Clip the values (keeping the imaginary part if it exists)
    # We create a copy to avoid modifying the original array in place
    clipped = np.clip(eloc_real, lower_bound, upper_bound)
    
    # If the original was complex, restore the imaginary part
    if np.iscomplexobj(eloc):
        return clipped + 1j * np.imag(eloc)
    return clipped  
    
def define_load_test_no_tau(wf,  numsamples,path_to_weights, Ee, clipped_e = False):
    state_dict = torch.load(path_to_weights, map_location=torch.device('cpu'))   
    new_state_dict = {}
    for key, value in state_dict.items():
        # Strip prefixes and rename keys to match current architecture
        new_key = key.replace('model.', '').replace('cell.', 'rnn.')
        new_state_dict[new_key] = value
    #  Load the RE-MAPPED weights
    wf.model.load_state_dict(new_state_dict, strict=False)
    print("Successfully remapped and loaded weights.")
    
    with torch.no_grad():
        test_samples_after = wf.sample_no_tau(numsamples)
        if clipped_e:
            # 1. Get raw energies
            raw_gs_after = J1J2_local_energies(wf, N, J1, J2, Bz, numsamples, test_samples_after, Marshall_sign=True)
    
            # 2. APPLY CLIPPING
            test_gs_after = clip_local_energies(raw_gs_after, threshold=5.0)
    
            # 3. Calculate statistics on cleaned data
            gs_mean_a = np.round(np.mean(test_gs_after), 4)
            gs_var_a = np.round(np.var(test_gs_after), 4)
    
            # Optional: Count how many were clipped to see if the model is unstable
            num_clipped = np.sum(np.real(raw_gs_after) != np.real(test_gs_after))
            print(f"Clipped {num_clipped} outlier samples out of {numsamples}")
        else:
            test_gs_after =  J1J2_local_energies(wf, N, J1, J2, Bz, numsamples, test_samples_after, Marshall_sign=True)
            gs_mean_a = np.round(np.mean(test_gs_after),4)
            gs_var_a = np.round(np.var(test_gs_after),4)
    
    #wf.model.summary()
    #print('====================================================================')
    print(f'After loading weights, the ground state energy mean and variance are:')
    print(f'Mean E = {gs_mean_a}, var E = {gs_var_a}')
    print(f'DMRG energy  is {np.round(Ee,4)}')

In [3]:
N=100
numsamples = 10000
units = 70
syssize = 100
J1_ = 1.0
J1 = +J1_ * np.ones(syssize)
Bz = +0.0 * np.ones(syssize)

## J2=0.8, sc=4.0

seed 333 didn't converge.

In [8]:
J2_ = 0.8
J2 = +J2_ * np.ones(syssize)
Ee_08=-42.07006297371643
spatial_clamp=4.0
fname='no_tau_J2=0.8'

In [20]:
seed=111
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=spatial_clamp, seed=seed)
h_wl = f'{fname}/J2={J2_}_sc={spatial_clamp}/seed={seed}/N100_J1=1.0|J2={J2_}_100_LorentzGRU_70_ns=80_MsTrue_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_08, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
Successfully remapped and loaded weights.
Clipped 589 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-40.28969955444336-0.012400000356137753j), var E = 1.6574000120162964
DMRG energy  is -42.0701
Time taken=1.697 hrs


In [21]:
seed=222
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=spatial_clamp, seed=seed)
h_wl = f'{fname}/J2={J2_}_sc={spatial_clamp}/seed={seed}/N100_J1=1.0|J2={J2_}_100_LorentzGRU_70_ns=80_MsTrue_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_08, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
Successfully remapped and loaded weights.
Clipped 301 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-40.364898681640625-0.010099999606609344j), var E = 1.993899941444397
DMRG energy  is -42.0701
Time taken=6.954 hrs


In [29]:
seed=444
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=spatial_clamp, seed=seed)
h_wl = f'{fname}/J2={J2_}_sc={spatial_clamp}/seed={seed}/N100_J1=1.0|J2={J2_}_100_LorentzGRU_70_ns=80_MsTrue_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_08, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
Successfully remapped and loaded weights.
Clipped 238 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-39.824798583984375-0.003700000001117587j), var E = 1.7236000299453735
DMRG energy  is -42.0701
Time taken=0.999 hrs


In [9]:
seed=555
wf=Lorentzwavefunction(systemsize=syssize, cell_type='LorentzGRU', units=units,spatial_clamp=spatial_clamp, seed=seed)
h_wl = f'{fname}/J2={J2_}_sc={spatial_clamp}/seed={seed}_retrained/N100_J1=1.0|J2={J2_}_100_LorentzGRU_70_ns=80_MsTrue_checkpoint.pt'
total_params = sum(p.numel() for p in wf.model.parameters())
print(f"Total Parameters: {total_params:,}")

t0 = time.time()
define_load_test_no_tau(wf, numsamples, h_wl, Ee_08, clipped_e=True)
t1 = time.time()
print(f'Time taken={np.round((t1-t0)/3600,3)} hrs')

Total Parameters: 15,615
Successfully remapped and loaded weights.
Clipped 95 outlier samples out of 10000
After loading weights, the ground state energy mean and variance are:
Mean E = (-41.532501220703125-0.0026000000070780516j), var E = 0.0812000036239624
DMRG energy  is -42.0701
Time taken=0.624 hrs
